# **Colab notebook fork from https://github.com/TheLastBen/fast-stable-diffusion. [ComfyUI Colab](https://colab.research.google.com/github/TheLastBen/fast-stable-diffusion/blob/main/fast_stable_diffusion_ComfyUI.ipynb)**

In [ ]:
#@markdown # Connect Google Drive
from google.colab import drive
from IPython.display import clear_output
import ipywidgets as widgets
import os

def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth));display(inf)
Shared_Drive = "" #@param {type:"string"}
#@markdown - Leave empty if you're not using a shared drive

print("[0;33mConnecting...")
drive.mount('/content/gdrive')

if Shared_Drive!="" and os.path.exists("/content/gdrive/Shareddrives"):
  mainpth="Shareddrives/"+Shared_Drive
else:
  mainpth="MyDrive"

clear_output()
inf('\u2714 Done','success', '50px')

In [ ]:
#@markdown # Install AUTOMATIC1111
from IPython.utils import capture
from IPython.display import clear_output
import ipywidgets as widgets
import os
import time
import base64

def inf(msg, style, wdth):
    btn = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth))
    display(btn)

blsaphemy = base64.b64decode(("ZWJ1aQ==").encode('ascii')).decode('ascii')

DRIVE = "/content/gdrive/MyDrive"
sd_path = f"{DRIVE}/sd"
webui_path = f"{sd_path}/stable-diffusion-w{blsaphemy}"
assets_path = f"{webui_path}/repositories/stable-diffusion-w{blsaphemy}-assets"

with capture.capture_output() as cap:
    os.makedirs(sd_path, exist_ok=True)

    os.environ['TRANSFORMERS_CACHE'] = f"{webui_path}/cache"
    os.environ['TORCH_HOME'] = f"{webui_path}/cache"

    # เช็คก่อน — ยังไม่ makedirs
    if not os.path.exists(webui_path):
        print("Cloning AUTOMATIC1111...")
        !git clone -q --branch master https://github.com/AUTOMATIC1111/stable-diffusion-w{blsaphemy} {webui_path}
        # makedirs หลัง clone
        os.makedirs(f"{webui_path}/cache", exist_ok=True)
        os.makedirs(f"{webui_path}/repositories", exist_ok=True)
    else:
        print("WebUI already exists, skipping clone.")

    if not os.path.exists(assets_path):
        print("Cloning assets...")
        !git clone -q https://github.com/AUTOMATIC1111/stable-diffusion-w{blsaphemy}-assets {assets_path}
    else:
        print("Assets already exists, skipping clone.")

    if not os.path.exists("/content/diffusers"):
        !git clone -q --depth 1 --branch main https://github.com/TheLastBen/diffusers /content/diffusers

clear_output()
inf('\u2714 Done', 'success', '50px')

#@markdown ---

In [ ]:
#@markdown # Install Requirements

print('[1;32mInstalling requirements...')

with capture.capture_output() as cap:
  !rm -r /usr/local/lib/python3.12/dist-packages/gradio*
  %cd /content/
  !wget -q -i https://raw.githubusercontent.com/TheLastBen/fast-stable-diffusion/main/Dependencies/A1111.txt
  !dpkg -i *.deb
  if not os.path.exists('/content/gdrive/'+mainpth+'/sd/stablediffusion'):
    !tar -C /content/gdrive/$mainpth --zstd -xf sd_mrep.tar.zst
  !tar -C / --zstd -xf gcolabdeps.tar.zst
  !rm -f *.deb *.zst *.txt
  if not os.path.exists('/content/gdrive/'+mainpth+'/sd/libtcmalloc/libtcmalloc_minimal.so.4'):
    %env CXXFLAGS=-std=c++14
    !wget -q https://github.com/gperftools/gperftools/releases/download/gperftools-2.5/gperftools-2.5.tar.gz && tar zxf gperftools-2.5.tar.gz && mv gperftools-2.5 gperftools
    !wget -q https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/Patch
    %cd /content/gperftools
    !patch -p1 < /content/Patch
    !./configure --enable-minimal --enable-libunwind --enable-frame-pointers --enable-dynamic-sized-delete-support --enable-sized-delete --enable-emergency-malloc; make -j4
    !mkdir -p /content/gdrive/$mainpth/sd/libtcmalloc && cp .libs/libtcmalloc*.so* /content/gdrive/$mainpth/sd/libtcmalloc
    %env LD_PRELOAD=/content/gdrive/$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4
    %cd /content
    !rm *.tar.gz Patch && rm -r /content/gperftools
  else:
    %env LD_PRELOAD=/content/gdrive/$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4

  !pip uninstall jax -y
  !pip install -U xformers
  !pip install wandb==0.15.12 pydantic==1.10.2 numpy==1.26 scipy==1.15.3 controlnet_aux --no-deps -qq
  !pip install diffusers accelerate -U --no-deps -qq
  !rm -r /usr/local/lib/python3.12/dist-packages/tensorflow*
  os.environ['PYTHONWARNINGS'] = 'ignore'
  !sed -i 's@text = _formatwarnmsg(msg)@text =\"\"@g' /usr/lib/python3.12/warnings.py
  !sed -i 's@from pytorch_lightning.loggers.wandb import WandbLogger  # noqa: F401@@g' /usr/local/lib/python3.12/dist-packages/pytorch_lightning/loggers/__init__.py
  !sed -i 's@from .mailbox import ContextCancelledError@@g' /usr/local/lib/python3.12/dist-packages/wandb/sdk/lib/retry.py
  !sed -i 's@raise ContextCancelledError("retry timeout")@print("retry timeout")@g' /usr/local/lib/python3.12/dist-packages/wandb/sdk/lib/retry.py
  !sed -i 's@globalns, localns, set()@globalns, localns, recursive_guard=set()@g' /usr/local/lib/python3.12/dist-packages/pydantic/typing.py

clear_output()
inf('\u2714 Done','success', '50px')

#@markdown ---

In [ ]:
import os
import glob
import re

def set_active_model(path):
    global model
    model = path

def detect_source(url):
    """Detect download source and return source name or None if unsupported"""
    if re.search(r'huggingface\.co', url):
        return 'huggingface'
    elif re.search(r'civitai\.(com|red)', url):
        return 'civitai'
    else:
        return None

def get_filename(url, source):
    """Extract filename from URL based on source"""
    if source == 'huggingface':
        # e.g. /resolve/main/model.safetensors
        match = re.search(r'/resolve/[^/]+/([^?#]+)', url)
        if match:
            return match.group(1).split('/')[-1]
    elif source == 'civitai':
        match = re.search(r'filename=([^&]+)', url)
        if match:
            return match.group(1)
        match = re.search(r'models/(\d+)', url)
        if match:
            return f"civitai_model_{match.group(1)}.safetensors"
    # Generic fallback
    return url.split('/')[-1].split('?')[0] or "custom_model.safetensors"

#@markdown # Model Download/Select

#@markdown ---
#@markdown **Select your download options (Select only one):**
mode = "use_recommended_model" #@param ["use_recommended_model", "use_exists_model", "download_custom_model"]

#@markdown **Paths and URLs:**
existing_model_path = "" #@param {type:"string"}
custom_model_url = "https://civitai.red/api/download/models/889818?fileId=2763986" #@param {type:"string"}

#@markdown **Civitai API Key (required for civitai.com, leave blank for civitai.red):**
civitai_api_key = "" #@param {type:"string"}

model_dir = "/content/gdrive/MyDrive/sd/stable-diffusion-webui/models/Stable-diffusion"
os.makedirs(model_dir, exist_ok=True)

# Recommended model details
rec_model_name = "waiillustriousSDXL_v170.safetensors"
rec_model_url = "https://civitai.red/api/download/models/2883731?type=Model&format=SafeTensor&size=pruned&fp=fp16"

if mode == "use_recommended_model":
    dest_path = os.path.join(model_dir, rec_model_name)
    if os.path.exists(dest_path):
        print(f"Recommended model '{rec_model_name}' already exists. Skipping download.")
    else:
        print(f"Downloading recommended model to {dest_path}...")
        !wget -c "{rec_model_url}" -O "{dest_path}"
    set_active_model(dest_path)

elif mode == "use_exists_model":
    target_path = existing_model_path
    if not target_path or not os.path.exists(target_path):
        print("No path provided or path invalid. Searching for first available model in model_dir...")
        found_models = glob.glob(os.path.join(model_dir, "*.safetensors")) + glob.glob(os.path.join(model_dir, "*.ckpt"))
        if found_models:
            target_path = found_models[0]
            print(f"Auto-selected model: {os.path.basename(target_path)}")
        else:
            print("No models found in folder.")
            target_path = None

    if target_path and os.path.exists(target_path):
        model_name = os.path.basename(target_path)
        dest = os.path.join(model_dir, model_name)
        if not os.path.exists(dest):
            print(f"Linking {model_name}...")
            !ln -s "{target_path}" "{dest}"
        else:
            print(f"Model {model_name} is ready.")
        set_active_model(target_path)

elif mode == "download_custom_model":
    if not custom_model_url:
        print("❌ No URL provided.")
    else:
        source = detect_source(custom_model_url)

        if source is None:
            print("❌ Unsupported source. Only HuggingFace (huggingface.co) and CivitAI (civitai.com / civitai.red) are allowed.")
        else:
            print(f"✅ Source detected: {source}")
            model_name = get_filename(custom_model_url, source)
            dest_path = os.path.join(model_dir, model_name)

            if source == 'civitai' and civitai_api_key and 'civitai.com' in custom_model_url:
                # Append API key for civitai.com
                sep = '&' if '?' in custom_model_url else '?'
                download_url = f"{custom_model_url}{sep}token={civitai_api_key}"
                print(f"Downloading from CivitAI (with API key): {model_name}")
            else:
                download_url = custom_model_url
                print(f"Downloading from {source}: {model_name}")

            !wget -c "{download_url}" -O "{dest_path}"
            set_active_model(dest_path)

In [ ]:
#@markdown # Start Stable-Diffusion
from IPython.utils import capture
import time
import sys
import fileinput
from pyngrok import ngrok, conf
import re


Ngrok_token = "" #@param {type:"string"}

#@markdown - Input your ngrok token if you want to use ngrok server

User = "" #@param {type:"string"}
Password= "" #@param {type:"string"}
#@markdown - Add credentials to your Gradio interface (optional)

auth=f"--gradio-auth {User}:{Password}"
if User =="" or Password=="":
  auth=""

with capture.capture_output() as cap:
  %cd /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/modules/
  !wget -q -O extras.py https://raw.githubusercontent.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy/master/modules/extras.py
  !wget -q -O sd_models.py https://raw.githubusercontent.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy/master/modules/sd_models.py
  !wget -q -O /usr/local/lib/python3.12/dist-packages/gradio/blocks.py https://raw.githubusercontent.com/TheLastBen/fast-stable-diffusion/main/AUTOMATIC1111_files/blocks.py
  %cd /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/

  !sed -i 's@shared.opts.data\["sd_model_checkpoint"] = checkpoint_info.title@shared.opts.data\["sd_model_checkpoint"] = checkpoint_info.title;model.half()@' /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/modules/sd_models.py
  !sed -i 's@ui.create_ui().*@ui.create_ui();shared.demo.queue(concurrency_count=999999,status_update_rate=0.1)@' /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/webui.py
  !sed -i "s@map_location='cpu'@map_location='cuda'@" /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/modules/extras.py

  !sed -i 's@possible_sd_paths =.*@possible_sd_paths = [\"/content/gdrive/MyDrive/sd/stablediffusion\"]@' /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/modules/paths.py
  !sed -i 's@\.\.\/@src/@g' /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/modules/paths.py
  !sed -i 's@src/generative-models@generative-models@g' /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/modules/paths.py

  !sed -i 's@print(\"No module.*@@' /content/gdrive/MyDrive/sd/stablediffusion/ldm/modules/diffusionmodules/model.py
  !sed -i 's@\["sd_model_checkpoint"\]@\["sd_model_checkpoint", "sd_vae", "CLIP_stop_at_last_layers", "inpainting_mask_weight", "initial_noise_multiplier"\]@g' /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/modules/shared.py
  !sed -i "s@res = self.CLIPTextModel_from_pretrained(None@res = self.CLIPTextModel_from_pretrained(pretrained_model_name_or_path@" /content/gdrive/MyDrive/sd/stable-diffusion-webui/modules/sd_disable_initialization.py

share=''
if Ngrok_token!="":
  ngrok.kill()
  srv=ngrok.connect(7860, pyngrok_config=conf.PyngrokConfig(auth_token=Ngrok_token) , bind_tls=True).public_url

  for line in fileinput.input('/usr/local/lib/python3.12/dist-packages/gradio/blocks.py', inplace=True):
    if line.strip().startswith('self.server_name ='):
        line = f'            self.server_name = "{srv[8:]}"\n'
    if line.strip().startswith('self.protocol = "https"'):
        line = '            self.protocol = "https"\n'
    if line.strip().startswith('if self.local_url.startswith("https") or self.is_colab'):
        line = ''
    if line.strip().startswith('else "http"'):
        line = ''
    sys.stdout.write(line)
else:
  share='--share'

ckptdir=''
if os.path.exists('/content/temp_models'):
  ckptdir='--ckpt-dir /content/temp_models'

try:
  model
  if os.path.isfile(model):
    !python /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/webui.py $share --api --disable-safe-unpickle --enable-insecure-extension-access --no-download-sd-model --no-half-vae  --ckpt "$model" --xformers $auth --disable-console-progressbars --skip-version-check $ckptdir
  else:
    !python /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/webui.py $share --api --disable-safe-unpickle --enable-insecure-extension-access --no-download-sd-model --no-half-vae  --ckpt-dir "$model" --xformers $auth --disable-console-progressbars --skip-version-check
except:
   !python /content/gdrive/MyDrive/sd/stable-diffusion-w$blsaphemy/webui.py $share --api --disable-safe-unpickle --enable-insecure-extension-access --no-download-sd-model --no-half-vae --xformers $auth --disable-console-progressbars --skip-version-check $ckptdir